In [132]:
import wget, os, gzip, pickle, random, re, sys, importlib, tqdm, math, os, gzip, re, string

from tqdm import trange
from collections import Counter

import torch

import random
from random import choice

IMDB_URL = 'http://dlvu.github.io/data/imdb.{}.pkl.gz'
IMDB_FILE = 'imdb.{}.pkl.gz'
WP_DATA = 'https://codeberg.org/pbm/former/raw/branch/master/data/enwik8.gz'

PAD, START, END, UNK = '.pad', '.start', '.end', '.unk'

SENT = '_s'
TOY = {
    '_s': ['_s _adv', '_np _vp', '_np _vp _prep _np', '_np _vp ( _prep _np )', '_np _vp _con _s','_np _vp ( _con _s )'],
    '_adv': ['briefly', 'quickly', 'impatiently'],
    '_np': ['a _noun', 'the _noun', 'a _adj _noun', 'the _adj _noun'],
    '_prep': ['on', 'with', 'to', 'for', 'at'],
    '_con': ['while', 'but'],
    '_noun': ['mouse', 'bunny', 'cat', 'dog', 'man', 'woman', 'person', 'bear', 'koala', 'judge', 'businessman',
        'businesswoman', 'lawyer', 'teacher', 'engineer'],
    '_vp': ['walked', 'walks', 'ran', 'runs', 'goes', 'went', 'hiked'],
    '_adj': ['short', 'quick', 'busy', 'nice', 'gorgeous', 'spectacular', 'reluctant', 'systematic', 'willowy', 'engaged', 'synthetic']
}

PRINTABLE = set(ord(c) for c in (string.digits + string.ascii_letters + string.punctuation + string.whitespace))

def cas(i):
    """
    Character-as-string. Filters out the ascii codes that aren't safe to print.

    :return:
    """
    assert i >= 0 and i < 256
    return '□' if i not in PRINTABLE else str(chr(i))

def t(blist):
    return torch.tensor([int(b) for b in blist], dtype=torch.uint8)

def gen_sentence(sent=SENT, g=TOY):

    symb = '_[a-z]*'

    while True:

        match = re.search(symb, sent)
        if match is None:
            return sent

        s = match.span()
        sent = sent[:s[0]] + random.choice(g[sent[s[0]:s[1]]]) + sent[s[1]:]

def load_toy(ntrain=100_000, ntest=20_000, to_torch=True, final=False, seed=0):
    """
    Generates language from a toy grammar.
    :param ntrain:
    :param ntest:
    :param to_torch: Whether to return torch tensors (if false, returns python lists)
    :param final: Whether to return the test set or the validation set (True for test)
    :return:
    """

    random.seed(seed)

    train, test = '', ''
    while len(train) < ntrain:
        train += gen_sentence() + ' . '

    random.seed(seed if final else seed + 1)
    # -- change the seed so we get different test/val sets depending on `final`

    while len(test) < ntest:
        test += gen_sentence() + ' . '

    ctr = Counter(train + test)
    i2t = [PAD, START, END, UNK] + [t for t, _ in ctr.most_common()]
    t2i = { w : i for  i, w in enumerate(i2t)}

    train = [t2i[t] for t in train]
    test  = [t2i[t] for t in test]
    
    if to_torch:
        return (t(train), t(test)), (i2t, t2i) # Torch vectors (this takes a few seconds)

    return (train, test), (i2t, t2i)

def load_wp(fname='enwik8.gz', split=(90, 5, 5), to_torch=True, final=False):
    """
    Load the enwik8 dataset from the Hutter challenge as a list or vector of bytes.

    :param fname: Filename for the downloaded data.
    :param split: Percentages for the train/val/test split.
    :param to_torch: Whether to return torch tensors (True) or python lists (False)
    :param final: If False, returns train/val if True returns train/test with the validation
    data added to the training data.
    :return:
    """

    if not os.path.exists(fname):
        # If it doesn't exist, download it
        print('Downloading')
        wget.download(WP_DATA, out=fname)
        
    with gzip.open(fname, 'r') if fname.endswith('.gz') else open(fname, 'rb') as file:

        all = file.read()
        ctr = Counter(all)

        i2t = {token : cas(token) for token, freq in ctr.most_common()}
        t2i = {w : i for i, w in enumerate(i2t)}

        split = tuple(s/sum(split) for s in split)
        split = tuple(int(s * len(all)) for s in split)

        train, val, test = all[:split[0]], all[split[0]:split[0]+split[1]], all[split[0]+split[1]:]

        if final:
            train = train + val
            wh = test
        else:
            wh = val

        if to_torch:
            return (t(train), t(wh)), (i2t, t2i)

        return (train, wh), (i2t, t2i)


def load_xor(ntrain=25_000, ntest=25_000, seed=0):

    random.seed(seed)

    i2w = [PAD, START, END, UNK, 'true', 'false'] #
    w2i = {w : i for i, w in enumerate(i2w)}

    dataset, labels = [], []
    for _ in range(ntrain + ntest):
        sentence = [
            choice((i2w[4], i2w[5])),
            choice((i2w[4], i2w[5]))
        ]

        f1, f2 = (sentence[0] == i2w[4]), (sentence[1] == i2w[4]) # true: very/great false: not/terrible
        # -- these words are the only meaningful features
        label = 0 if f1 != f2 else 1

        dataset.append([w2i[word] for word in sentence])
        labels.append(label)

    return \
        (dataset[:ntrain], labels[:ntrain]), \
        (dataset[ntrain:], labels[ntrain:]), \
        (i2w, w2i), 2

def load_imdb_synth(ntrain=25_000, ntest=25_000, seed=0):
    """
    Synthetic IMDb dataset
    :param seed:
    :param voc:
    :return:
    """

    random.seed(seed)

    adjectives = ['classic', 'silent', 'modern', 'vintage', 'independent', 'foreign', 'animated', 'documentary',
    'epic', 'dramatic', 'romantic', 'comic', 'thrilling', 'mysterious', 'gritty', 'stylized', 'iconic', 'acclaimed',
    'popular', 'forgettable', 'unreleased', 'awardwinning', 'blockbuster', 'lowbudget', 'highbudget', 'experimental',
    'mainstream', 'cult', 'notable', 'original']
    nouns = ['movie', 'film', 'motion-picture', 'feature', 'featurette', 'picture', 'flick', 'cinema', 'screenplay',
    'blockbuster', 'talkie', 'silent', 'biopic', 'short', 'docudrama', 'documentary', 'animation', 'cartoon',
    'anime', 'telefilm', 'miniseries', 'drama', 'comedy', 'thriller', 'western', 'musical', 'noir']
    verbs = ['was', 'is', 'became', 'becomes', 'seemed', 'seems']

    i2w = [PAD, START, END, UNK, 'this', 'not', 'very', 'great','terrible'] + verbs + adjectives + nouns
    w2i = {w : i for i, w in enumerate(i2w)}

    dataset, labels = [], []
    for _ in range(ntrain + ntest):
        sentence = [
            i2w[4], # this
            choice(adjectives), # old
            choice(nouns), # movie
            choice(verbs), # was
            choice((i2w[5], i2w[6])),
            choice((i2w[7], i2w[8]))
        ]

        f1, f2 = (sentence[4] == i2w[6]), (sentence[5] == i2w[7]) # true: very/great false: not/terrible
        # -- these words are the only meaningful features
        label = 0 if f1 != f2 else 1

        dataset.append([w2i[word] for word in sentence])
        labels.append(label)

    return \
        (dataset[:ntrain], labels[:ntrain]), \
        (dataset[ntrain:], labels[ntrain:]), \
        (i2w, w2i), 2


def load_imdb(final=False, val=5000, seed=0, voc=None, char=False):

    cst = 'char' if char else 'word'

    imdb_url = IMDB_URL.format(cst)
    imdb_file = IMDB_FILE.format(cst)

    if not os.path.exists(imdb_file):
        wget.download(imdb_url)

    with gzip.open(imdb_file) as file:
        sequences, labels, i2w, w2i = pickle.load(file)

    if voc is not None and voc < len(i2w):
        nw_sequences = {}

        i2w = i2w[:voc]
        w2i = {w: i for i, w in enumerate(i2w)}

        mx, unk = voc, w2i['.unk']
        for key, seqs in sequences.items():
            nw_sequences[key] = []
            for seq in seqs:
                seq = [s if s < mx else unk for s in seq]
                nw_sequences[key].append(seq)

        sequences = nw_sequences

    if final:
        return (sequences['train'], labels['train']), (sequences['test'], labels['test']), (i2w, w2i), 2

    # Make a validation split
    random.seed(seed)

    x_train, y_train = [], []
    x_val, y_val = [], []

    val_ind = set( random.sample(range(len(sequences['train'])), k=val) )
    for i, (s, l) in enumerate(zip(sequences['train'], labels['train'])):
        if i in val_ind:
            x_val.append(s)
            y_val.append(l)
        else:
            x_train.append(s)
            y_train.append(l)

    return (x_train, y_train), \
           (x_val, y_val), \
           (i2w, w2i), 2


In [133]:
(x_train, y_train), (x_val, y_val), (i2w, w2i), numcls = load_imdb(final=False)

print([i2w[w] for w in x_train[142]])
print(f'.pad index: {w2i[".pad"]}')

print(x_train[0])

['ronald', 'colman', 'gives', 'a', 'terrific', 'performance', 'as', 'a', 'stage', 'actor', 'who', 'really', 'gets', 'into', 'his', 'work', 'when', 'he', 'plays', 'othello', 'on', 'the', 'stage', 'he', 'takes', 'on', 'the', 'persona', 'with', 'dire', 'results', 'good', 'film', 'with', 'a', 'great', 'supporting', 'cast', 'well', 'worth', 'watching']
.pad index: 0
[14, 19, 9, 379, 22, 11, 50, 52, 53, 290]


In [134]:
# find size of the embedding space
flat_list = [
    item
    for inner in x_train 
    for item in inner
]

num_embeddings = max(flat_list)
print(num_embeddings)

99429


In [135]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import zip_longest

batch_size = 100

def create_batches(word_list: list, label_list: list, batch_size: int = 100):
    """Create padded batches of a list."""

    # sort list of reviews by length of the review, from short to long reviews.
    # simultaneousy sort list of labels in the same way. return as list.
    sorted_words, sorted_labels = (
        list(t)
        for t in 
        zip(*sorted(zip(word_list, label_list), key = lambda pair: len(pair[0])))
    )

    word_batches = []
    label_batches = []

    for i in range(0, len(sorted_words), batch_size):
        words_slice = sorted_words[i : i + batch_size]
        labels_slice = sorted_labels[i : i + batch_size]
        
        # pad list and make it np array
        padded_array = np.array(list(zip_longest(*words_slice, fillvalue=0))).T

        # convert np array to tensor and append
        words_batch = torch.tensor(padded_array, dtype=torch.long)
        labels_batch = torch.tensor(labels_slice)

        word_batches.append(words_batch)
        label_batches.append(labels_batch)

    return word_batches, label_batches
    
x_batches, y_batches = create_batches(x_train, y_train)
x_val_batches, y_val_batches = create_batches(x_val, y_val)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings + 1, embedding_dim=300)
        self.linear = nn.Linear(300, 2)

    def forward(self, x):
        # batch of words -> embedding layer
        embedded = self.embedding(x)

        # embedding layer -> max pooled along time dimension
        maxpooled = torch.max(embedded, 1).values
        meaned = torch.mean(embedded, 1)
        selected = embedded[:, 0, :]

        # outputs = logits
        logits = self.linear(meaned)

        return logits

net = Net()
optimizer = optim.Adam(net.parameters())
running_loss = 0.0

net.train()
for epoch in range(5):
    for i, (x_batch, y_batch) in enumerate(zip(x_batches, y_batches)):
        optimizer.zero_grad()
        output = net(x_batch)

        loss = F.cross_entropy(output, y_batch)

        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 10 == 9:    # print every 10 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0


[1,    10] loss: 0.685
[1,    20] loss: 0.676
[1,    30] loss: 0.681
[1,    40] loss: 0.667
[1,    50] loss: 0.663
[1,    60] loss: 0.652
[1,    70] loss: 0.643
[1,    80] loss: 0.643
[1,    90] loss: 0.629
[1,   100] loss: 0.627
[1,   110] loss: 0.624
[1,   120] loss: 0.620
[1,   130] loss: 0.611
[1,   140] loss: 0.612
[1,   150] loss: 0.596
[1,   160] loss: 0.586
[1,   170] loss: 0.587
[1,   180] loss: 0.584
[1,   190] loss: 0.567
[1,   200] loss: 0.835
[2,    10] loss: 0.525
[2,    20] loss: 0.504
[2,    30] loss: 0.520
[2,    40] loss: 0.491
[2,    50] loss: 0.494
[2,    60] loss: 0.461
[2,    70] loss: 0.472
[2,    80] loss: 0.468
[2,    90] loss: 0.455
[2,   100] loss: 0.464
[2,   110] loss: 0.466
[2,   120] loss: 0.472
[2,   130] loss: 0.457
[2,   140] loss: 0.475
[2,   150] loss: 0.448
[2,   160] loss: 0.443
[2,   170] loss: 0.454
[2,   180] loss: 0.459
[2,   190] loss: 0.441
[2,   200] loss: 0.629
[3,    10] loss: 0.349
[3,    20] loss: 0.354
[3,    30] loss: 0.378
[3,    40] 

In [ ]:
correct = 0
total = 0

net.eval()
with torch.no_grad():
    for x_batch, y_batch in zip(x_val_batches, y_val_batches):
        outputs = net(x_batch)
        
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print(f'Accuracy of the network on the 5000 validation reviews: {100 * correct // total} %')

Accuracy of the network on the 5000 validation reviews: 87 %


In [143]:
net_synth = Net()

optimizer = optim.Adam(net_synth.parameters())
running_loss = 0.0

(synth_x_train, synth_y_train), (synth_x_test, synth_y_test), (i2w, w2i), numcls = load_imdb_synth()

synth_x_batches, synth_y_batches = create_batches(synth_x_train, synth_y_train)
synth_x_val_batches, synth_y_val_batches = create_batches(synth_x_test, synth_y_test)

net_synth.train()
for epoch in range(5):
    for i, (x_batch, y_batch) in enumerate(zip(synth_x_batches, synth_y_batches)):
        optimizer.zero_grad()
        output = net_synth(x_batch)

        loss = F.cross_entropy(output, y_batch)

        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 10 == 9:    # print every 10 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0

correct = 0
total = 0

net_synth.eval()
with torch.no_grad():
    for x_batch, y_batch in zip(x_val_batches, y_val_batches):
        outputs = net_synth(x_batch)
        
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print(f'Accuracy of the network on the 5000 validation reviews: {100 * correct // total} %')

[1,    10] loss: 0.702
[1,    20] loss: 0.694
[1,    30] loss: 0.698
[1,    40] loss: 0.699
[1,    50] loss: 0.697
[1,    60] loss: 0.697
[1,    70] loss: 0.696
[1,    80] loss: 0.691
[1,    90] loss: 0.698
[1,   100] loss: 0.696
[1,   110] loss: 0.698
[1,   120] loss: 0.698
[1,   130] loss: 0.696
[1,   140] loss: 0.696
[1,   150] loss: 0.697
[1,   160] loss: 0.697
[1,   170] loss: 0.694
[1,   180] loss: 0.695
[1,   190] loss: 0.700
[1,   200] loss: 0.695
[1,   210] loss: 0.699
[1,   220] loss: 0.696
[1,   230] loss: 0.693
[1,   240] loss: 0.696
[1,   250] loss: 0.699
[2,    10] loss: 0.697
[2,    20] loss: 0.694
[2,    30] loss: 0.693
[2,    40] loss: 0.696
[2,    50] loss: 0.696
[2,    60] loss: 0.696
[2,    70] loss: 0.694
[2,    80] loss: 0.690
[2,    90] loss: 0.696
[2,   100] loss: 0.698
[2,   110] loss: 0.697
[2,   120] loss: 0.696
[2,   130] loss: 0.696
[2,   140] loss: 0.696
[2,   150] loss: 0.695
[2,   160] loss: 0.698
[2,   170] loss: 0.695
[2,   180] loss: 0.695
[2,   190] 

In [142]:
net_xor = Net()

optimizer = optim.Adam(net_xor.parameters())
running_loss = 0.0

(xor_x_train, xor_y_train), (xor_x_test, xor_y_test), (i2w, w2i), numcls = load_xor()

xor_x_batches, xor_y_batches = create_batches(xor_x_train, xor_y_train)
xor_x_val_batches, xor_y_val_batches = create_batches(xor_x_test, xor_y_test)

net_xor.train()
for epoch in range(5):
    for i, (x_batch, y_batch) in enumerate(zip(synth_x_batches, synth_y_batches)):
        optimizer.zero_grad()
        output = net_xor(x_batch)

        loss = F.cross_entropy(output, y_batch)

        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 10 == 9:    # print every 10 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0

correct = 0
total = 0

net_xor.eval()
with torch.no_grad():
    for x_batch, y_batch in zip(x_val_batches, y_val_batches):
        outputs = net_xor(x_batch)
        
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print(f'Accuracy of the network on the 5000 validation reviews: {100 * correct // total} %')

[1,    10] loss: 0.695
[1,    20] loss: 0.700
[1,    30] loss: 0.701
[1,    40] loss: 0.696
[1,    50] loss: 0.700
[1,    60] loss: 0.699
[1,    70] loss: 0.697
[1,    80] loss: 0.691
[1,    90] loss: 0.694
[1,   100] loss: 0.700
[1,   110] loss: 0.699
[1,   120] loss: 0.697
[1,   130] loss: 0.699
[1,   140] loss: 0.696
[1,   150] loss: 0.695
[1,   160] loss: 0.698
[1,   170] loss: 0.694
[1,   180] loss: 0.694
[1,   190] loss: 0.698
[1,   200] loss: 0.695
[1,   210] loss: 0.698
[1,   220] loss: 0.697
[1,   230] loss: 0.693
[1,   240] loss: 0.697
[1,   250] loss: 0.698
[2,    10] loss: 0.696
[2,    20] loss: 0.694
[2,    30] loss: 0.693
[2,    40] loss: 0.696
[2,    50] loss: 0.696
[2,    60] loss: 0.696
[2,    70] loss: 0.694
[2,    80] loss: 0.690
[2,    90] loss: 0.695
[2,   100] loss: 0.698
[2,   110] loss: 0.697
[2,   120] loss: 0.696
[2,   130] loss: 0.697
[2,   140] loss: 0.696
[2,   150] loss: 0.695
[2,   160] loss: 0.698
[2,   170] loss: 0.695
[2,   180] loss: 0.695
[2,   190] 